# Evaluation des checkpoints sur crues completes

Ce notebook evalue les checkpoints sur les crues completes `0-0-80`.


In [ ]:
import math
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import torch
from matplotlib.colors import ListedColormap

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset, TelemacDatasetWithQ
from python.eval_rollout import (
    build_model as _build_rollout_model,
    build_rollout_context,
    create_rollout_state,
    denormalize_delta,
    denormalize_state,
    load_model_checkpoint as _load_rollout_model_checkpoint,
    rollout_step,
)
from python.python_code.data_manip.extraction.telemac_file import TelemacFile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.dpi'] = 120
device


In [ ]:
# =====================
# Parametres utilisateur
# =====================
RAW_DYNAMIC_ROOT = '/work/m24046/m24046mrcr/dataset_x8_avec_ts/fullx8/'
HYDRO_ROOT = '/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes'
DATA_DIRS = {
    'normal': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_base.bin',
    'multimesh': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Multimesh_8_32.bin',
}

TEST_EVENTS = [
    'Group_1_peak_2600',
    'Group_2_peak_1000',
    'Group_2_peak_1200',
    'Group_2_peak_1600',
    'Group_4_peak_2000',
    'Group_1_peak_1200',
    'Group_1_peak_2400',
    'Group_3_peak_3400',
    'Group_1_peak_1400',
    'Group_1_peak_2000',
    'Group_1_peak_2200',
    'Group_2_peak_3600',
    'Group_3_peak_2200',
    'Group_3_peak_2800',
    'Group_4_peak_1200',
    'Group_4_peak_3000',
]

def build_dynamic_path(event_name):
    return os.path.join(RAW_DYNAMIC_ROOT, f'{event_name}_{event_name}_0_0-80_interpolated.pkl')

def build_hydro_path(event_name):
    return os.path.join(HYDRO_ROOT, f'generated_hydrographs_{event_name}.liq')

METHODS = [
    {
        'name': 'Experiment 5',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience8/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'multimesh',
    },
    {
        'name': 'Experiment 6',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience10/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'multimesh',
    },
]
RUNS = {method['name']: f"{method['name']}@{method['epoch']}" for method in METHODS}
METHOD_BY_NAME = {method['name']: method for method in METHODS}

EXPERIENCE_COLORS = {
    'Experiment 5': '#9467bd',
    'Experiment 6': '#8c564b',
}

DT_SECONDS = 1800.0
REQUESTED_MAX_STEPS = 80
THRESHOLD_M = 0.1

NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

QUAL_EVENT_INDEX = -3
QUAL_SEQUENCE_INDEX = 0
QUAL_THRESHOLD_M = THRESHOLD_M
QUAL_HOURS = [6, 12, 24, 36]
X8_MESH_PATH = '/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_corrige.slf'


In [ ]:
def get_dynamic_length(file_path):
    with open(file_path, 'rb') as fp:
        return len(pickle.load(fp))

dynamic_lengths = [get_dynamic_length(build_dynamic_path(event_name)) for event_name in TEST_EVENTS]
min_num_samples = min(dynamic_lengths)
AVAILABLE_STEPS = max(1, min(REQUESTED_MAX_STEPS, min_num_samples))
HORIZONS_STEPS = list(range(1, AVAILABLE_STEPS + 1))
HORIZON_HOURS = np.asarray(HORIZONS_STEPS, dtype=float) * DT_SECONDS / 3600.0
SEQUENCE_LENGTH = AVAILABLE_STEPS

print(
    f'{len(TEST_EVENTS)} crues test | sequence_length={SEQUENCE_LENGTH} '
    f'| horizons={HORIZONS_STEPS[0]}..{HORIZONS_STEPS[-1]}'
)


In [ ]:
# =====================
# Evaluation
# =====================
def build_model(num_input_features):
    return _build_rollout_model(
        num_input_features=num_input_features,
        num_edge_features=NUM_EDGE_FEATURES,
        num_output_features=NUM_OUTPUT_FEATURES,
        mp_layers=MP_LAYERS,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def build_event_dataset(event_name, ckpt_dir, use_q_feature, mesh, sequence_length, split='test'):
    dynamic_file = build_dynamic_path(event_name)
    data_dir = DATA_DIRS[mesh]

    if use_q_feature:
        return TelemacDatasetWithQ(
            name=f'eval_{event_name}',
            data_dir=data_dir,
            dynamic_data_files=[dynamic_file],
            hydro_data_files=[build_hydro_path(event_name)],
            split=split,
            ckpt_path=ckpt_dir,
            normalize=True,
            sequence_length=sequence_length,
            overlap=0,
            dt_seconds=DT_SECONDS,
        )

    return TelemacDataset(
        name=f'eval_{event_name}',
        data_dir=data_dir,
        dynamic_data_files=[dynamic_file],
        split=split,
        ckpt_path=ckpt_dir,
        normalize=True,
        sequence_length=sequence_length,
        overlap=0,
    )

def csi_from_binary(pred_mask, gt_mask):
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    denom = tp + fp + fn
    return float(tp / denom) if denom > 0 else math.nan

def evaluate_single_event(model, ds, horizons_steps, use_q_feature, threshold):
    if len(ds) == 0:
        raise ValueError('Dataset vide pour cet evenement.')

    # On garde le premier unrolling pour comparer toutes les crues sur la meme longueur commune.
    graphs = ds[0]
    max_h = max(horizons_steps)
    if len(graphs) < max_h:
        raise ValueError(f'Sequence trop courte: len(graphs)={len(graphs)} | max_h={max_h}')

    rollout_context = build_rollout_context(ds, device=device, use_q_feature=use_q_feature)
    step_to_index = {step: idx for idx, step in enumerate(horizons_steps)}

    l1 = np.full((len(horizons_steps), 3), np.nan, dtype=float)
    csi = np.full(len(horizons_steps), np.nan, dtype=float)
    rollout_state = create_rollout_state(graphs[0], rollout_context)

    for t in range(max_h):
        g_ref_t = graphs[t]
        x_ref_full_n = g_ref_t.ndata['x'][:, rollout_context.dyn_start:rollout_context.dyn_start + rollout_context.dyn_len].to(device)
        x_ref_t = denormalize_state(x_ref_full_n[:, :3], rollout_context)
        y_gt_n = g_ref_t.ndata['y'][:, :3].to(device)
        y_gt = denormalize_delta(y_gt_n, rollout_context)
        x_gt = x_ref_t + y_gt

        if use_q_feature and t + 1 < len(graphs):
            q_t1_n = graphs[t + 1].ndata['x'][:, rollout_context.dyn_start + 3:rollout_context.dyn_start + 4].to(device)
        elif use_q_feature:
            q_t1_n = rollout_state.xn_t_full[:, 3:4]
        else:
            q_t1_n = None

        step_result = rollout_step(
            model,
            rollout_state,
            rollout_context,
            x_gt=x_gt,
            q_t1_n=q_t1_n,
        )

        step = t + 1
        if step in step_to_index:
            step_idx = step_to_index[step]
            l1[step_idx] = torch.mean(
                torch.abs(step_result.predicted_state - step_result.target_state),
                dim=0,
            ).detach().cpu().numpy()
            h_pred = step_result.predicted_state[:, 0].detach().cpu().numpy()
            h_gt = step_result.target_state[:, 0].detach().cpu().numpy()
            csi[step_idx] = csi_from_binary(h_pred >= threshold, h_gt >= threshold)

        rollout_state = step_result.next_state

    return {'l1': l1, 'csi': csi}

def aggregate_event_metrics(event_metrics):
    l1_stack = np.stack([payload['l1'] for payload in event_metrics.values()], axis=0)
    csi_stack = np.stack([payload['csi'] for payload in event_metrics.values()], axis=0)
    return {
        'n_events': len(event_metrics),
        'l1_mean': np.nanmean(l1_stack, axis=0),
        'l1_std': np.nanstd(l1_stack, axis=0),
        'csi_mean': np.nanmean(csi_stack, axis=0),
        'csi_std': np.nanstd(csi_stack, axis=0),
    }

def evaluate_fixed_methods(methods, events, horizons_steps, threshold):
    fixed_metrics = {}

    for method in methods:
        run_name = f"{method['name']}@{method['epoch']}"
        print(f'Evaluation de {run_name}...')
        event_metrics = {}
        model = None

        for event_name in events:
            ds = build_event_dataset(
                event_name=event_name,
                ckpt_dir=method['ckpt_dir'],
                use_q_feature=method['use_q_feature'],
                mesh=method['mesh'],
                sequence_length=SEQUENCE_LENGTH,
            )

            if model is None:
                num_input_features = ds.base_graph.ndata['static'].shape[1] + 4
                model = build_model(num_input_features)
                _load_rollout_model_checkpoint(model, method['ckpt_dir'], method['epoch'], device=device)

            event_metrics[event_name] = evaluate_single_event(
                model,
                ds,
                horizons_steps=horizons_steps,
                use_q_feature=method['use_q_feature'],
                threshold=threshold,
            )

        fixed_metrics[run_name] = {
            'method': method['name'],
            'mesh': method['mesh'],
            'epoch': method['epoch'],
            'hours': np.asarray(horizons_steps, dtype=float) * DT_SECONDS / 3600.0,
            'summary': aggregate_event_metrics(event_metrics),
        }

        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return fixed_metrics


In [ ]:
fixed_metrics = evaluate_fixed_methods(
    METHODS,
    events=TEST_EVENTS,
    horizons_steps=HORIZONS_STEPS,
    threshold=THRESHOLD_M,
)


In [ ]:
# =====================
# Courbes mean ± std
# =====================
def _style_for_experience(label):
    return {
        'color': EXPERIENCE_COLORS[label],
        'marker': 'o',
        'linewidth': 2,
        'markersize': 5,
    }

def _plot_series_with_band(ax, hours, mean_vals, std_vals, label, use_log=False):
    lower = mean_vals - std_vals
    upper = mean_vals + std_vals

    if use_log:
        mean_vals = mean_vals.copy()
        lower = lower.copy()
        upper = upper.copy()
        mean_vals[mean_vals <= 0.0] = np.nan
        lower[lower <= 0.0] = np.nan
        upper[upper <= 0.0] = np.nan

    style = _style_for_experience(label)
    ax.plot(hours, mean_vals, label=label, **style)
    ax.fill_between(hours, lower, upper, color=style['color'], alpha=0.18, linewidth=0)

def plot_full_floods_csi(fixed_metrics):
    payload_5 = fixed_metrics[RUNS['Experiment 5']]
    payload_6 = fixed_metrics[RUNS['Experiment 6']]
    hours = payload_5['hours']

    fig, ax = plt.subplots(figsize=(11, 4.5))
    _plot_series_with_band(
        ax,
        hours,
        payload_5['summary']['csi_mean'],
        payload_5['summary']['csi_std'],
        'Experiment 5',
    )
    _plot_series_with_band(
        ax,
        hours,
        payload_6['summary']['csi_mean'],
        payload_6['summary']['csi_std'],
        'Experiment 6',
    )
    ax.set_title(f'CSI mean ± std on full floods (threshold={THRESHOLD_M:.3f} m)')
    ax.set_xlabel('Horizon (hours)')
    ax.set_ylabel('CSI')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

def plot_full_floods_l1(fixed_metrics):
    payload_5 = fixed_metrics[RUNS['Experiment 5']]
    payload_6 = fixed_metrics[RUNS['Experiment 6']]
    hours = payload_5['hours']

    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

    ax_h = axes[0]
    _plot_series_with_band(ax_h, hours, payload_5['summary']['l1_mean'][:, 0], payload_5['summary']['l1_std'][:, 0], 'Experiment 5')
    _plot_series_with_band(ax_h, hours, payload_6['summary']['l1_mean'][:, 0], payload_6['summary']['l1_std'][:, 0], 'Experiment 6')
    ax_h.set_title('L1 mean ± std on full floods')
    ax_h.set_ylabel('L1 h (m)')
    ax_h.grid(True, alpha=0.3)
    ax_h.legend(frameon=False)

    ax_u = axes[1]
    _plot_series_with_band(ax_u, hours, payload_5['summary']['l1_mean'][:, 1], payload_5['summary']['l1_std'][:, 1], 'Experiment 5', use_log=True)
    _plot_series_with_band(ax_u, hours, payload_6['summary']['l1_mean'][:, 1], payload_6['summary']['l1_std'][:, 1], 'Experiment 6', use_log=True)
    ax_u.set_ylabel(r'L1 u (m s$^{-1}$)')
    ax_u.set_yscale('log')
    ax_u.grid(True, alpha=0.3)

    ax_v = axes[2]
    _plot_series_with_band(ax_v, hours, payload_5['summary']['l1_mean'][:, 2], payload_5['summary']['l1_std'][:, 2], 'Experiment 5', use_log=True)
    _plot_series_with_band(ax_v, hours, payload_6['summary']['l1_mean'][:, 2], payload_6['summary']['l1_std'][:, 2], 'Experiment 6', use_log=True)
    ax_v.set_ylabel(r'L1 v (m s$^{-1}$)')
    ax_v.set_xlabel('Horizon (hours)')
    ax_v.set_yscale('log')
    ax_v.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

plot_full_floods_csi(fixed_metrics)
plot_full_floods_l1(fixed_metrics)


In [ ]:
# =====================
# Cartes binaires sur le maillage x8
# =====================
_BINARY_MODEL_CACHE = {}
_TRI_CACHE = {}

def hours_to_steps(hours):
    return int(round(hours * 3600.0 / DT_SECONDS))

def get_cached_model(ds, method):
    key = (method['ckpt_dir'], method['epoch'], method['mesh'])
    if key not in _BINARY_MODEL_CACHE:
        num_input_features = ds.base_graph.ndata['static'].shape[1] + 4
        model = build_model(num_input_features)
        _load_rollout_model_checkpoint(model, method['ckpt_dir'], method['epoch'], device=device)
        _BINARY_MODEL_CACHE[key] = model
    return _BINARY_MODEL_CACHE[key]

def get_tri2d(mesh_path):
    if mesh_path not in _TRI_CACHE:
        mesh = TelemacFile(mesh_path)
        x = np.asarray(mesh.meshx[:mesh.npoin2], dtype=np.float64)
        y = np.asarray(mesh.meshy[:mesh.npoin2], dtype=np.float64)

        if hasattr(mesh, 'ikle2') and mesh.ikle2 is not None:
            triangles = np.asarray(mesh.ikle2, dtype=np.int64)
            if triangles.min() == 1:
                triangles = triangles - 1
        else:
            triangles = np.asarray(mesh.tri.triangles, dtype=np.int64)

        _TRI_CACHE[mesh_path] = mtri.Triangulation(x, y, triangles)
    return _TRI_CACHE[mesh_path]

def node_binary_to_face_binary(triangles, node_binary):
    return node_binary[triangles].max(axis=1).astype(float)

def rollout_binary_maps_x8_for_method(method, event_index, sequence_index, horizon_steps, threshold):
    event_name = TEST_EVENTS[event_index]
    ds = build_event_dataset(
        event_name=event_name,
        ckpt_dir=method['ckpt_dir'],
        use_q_feature=method['use_q_feature'],
        mesh=method['mesh'],
        sequence_length=SEQUENCE_LENGTH,
    )
    graphs = ds[sequence_index]
    max_h = max(horizon_steps)
    model = get_cached_model(ds, method)
    rollout_context = build_rollout_context(ds, device=device, use_q_feature=True)
    rollout_state = create_rollout_state(graphs[0], rollout_context)
    captured = {}

    for t in range(max_h):
        g_ref_t = graphs[t]
        x_ref_full_n = g_ref_t.ndata['x'][:, rollout_context.dyn_start:rollout_context.dyn_start + rollout_context.dyn_len].to(device)
        x_ref_t = denormalize_state(x_ref_full_n[:, :3], rollout_context)
        y_gt_n = g_ref_t.ndata['y'][:, :3].to(device)
        y_gt = denormalize_delta(y_gt_n, rollout_context)
        x_gt = x_ref_t + y_gt

        if t + 1 < len(graphs):
            q_t1_n = graphs[t + 1].ndata['x'][:, rollout_context.dyn_start + 3:rollout_context.dyn_start + 4].to(device)
        else:
            q_t1_n = rollout_state.xn_t_full[:, 3:4]

        step_result = rollout_step(
            model,
            rollout_state,
            rollout_context,
            x_gt=x_gt,
            q_t1_n=q_t1_n,
        )

        step = t + 1
        if step in horizon_steps:
            h_pred = step_result.predicted_state[:, 0].detach().cpu().numpy()
            h_gt = step_result.target_state[:, 0].detach().cpu().numpy()
            pred_binary = h_pred >= threshold
            gt_binary = h_gt >= threshold
            captured[step] = {
                'prediction_binary_nodes': pred_binary,
                'reference_binary_nodes': gt_binary,
                'csi': csi_from_binary(pred_binary, gt_binary),
            }

        rollout_state = step_result.next_state

    return {'event_name': event_name, 'payloads': captured}

def plot_binary_maps_x8():
    horizon_steps = [hours_to_steps(hour) for hour in QUAL_HOURS]
    tri = get_tri2d(X8_MESH_PATH)
    cmap = ListedColormap(['white', '#1f4e79'])

    result_5 = rollout_binary_maps_x8_for_method(
        METHOD_BY_NAME['Experiment 5'],
        event_index=QUAL_EVENT_INDEX,
        sequence_index=QUAL_SEQUENCE_INDEX,
        horizon_steps=horizon_steps,
        threshold=QUAL_THRESHOLD_M,
    )
    result_6 = rollout_binary_maps_x8_for_method(
        METHOD_BY_NAME['Experiment 6'],
        event_index=QUAL_EVENT_INDEX,
        sequence_index=QUAL_SEQUENCE_INDEX,
        horizon_steps=horizon_steps,
        threshold=QUAL_THRESHOLD_M,
    )

    available_steps = [
        step for step in horizon_steps
        if step in result_5['payloads'] and step in result_6['payloads']
    ]
    fig, axes = plt.subplots(3, len(available_steps), figsize=(4.2 * len(available_steps), 9), squeeze=False)

    for col, step in enumerate(available_steps):
        gt_face = node_binary_to_face_binary(
            tri.triangles,
            result_5['payloads'][step]['reference_binary_nodes'],
        )
        exp5_face = node_binary_to_face_binary(
            tri.triangles,
            result_5['payloads'][step]['prediction_binary_nodes'],
        )
        exp6_face = node_binary_to_face_binary(
            tri.triangles,
            result_6['payloads'][step]['prediction_binary_nodes'],
        )

        ax = axes[0, col]
        ax.tripcolor(tri, facecolors=gt_face, shading='flat', cmap=cmap, vmin=0, vmax=1)
        ax.set_title(f'GT | {step * DT_SECONDS / 3600.0:.1f} h')
        ax.set_aspect('equal')
        ax.set_axis_off()

        ax = axes[1, col]
        ax.tripcolor(tri, facecolors=exp5_face, shading='flat', cmap=cmap, vmin=0, vmax=1)
        ax.set_title(f'Exp 5 | CSI={result_5["payloads"][step]["csi"]:.3f}')
        ax.set_aspect('equal')
        ax.set_axis_off()

        ax = axes[2, col]
        ax.tripcolor(tri, facecolors=exp6_face, shading='flat', cmap=cmap, vmin=0, vmax=1)
        ax.set_title(f'Exp 6 | CSI={result_6["payloads"][step]["csi"]:.3f}')
        ax.set_aspect('equal')
        ax.set_axis_off()

    fig.suptitle(
        f'{result_5["event_name"]} | threshold={100 * QUAL_THRESHOLD_M:.0f} cm',
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

plot_binary_maps_x8()
